<a href="https://colab.research.google.com/github/moucheng2017/my_simsiam/blob/main/notebooks/hierarchical-kway-vmf-ot-cifar10-ssl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hierarchical K-way vMF/OT Self-Labeling on Google Colab (CIFAR-10)

This notebook launches the **HiRef-primed K-way** recursive spherical optimal-transport experiment. 
Instead of a fixed binary tree, each level splits into `r_t` children, where the per-level branching 
factors `(r_1, ..., r_kappa)` are the rank-annealing prior from *Hierarchical Refinement: Optimal Transport 
to Infinity and Beyond* (Halmos et al., ICML 2025) -- coarse small ranks early, larger ranks later. 
Provide `rank_schedule` explicitly, or set it to `None` to derive the optimal schedule from 
`num_leaf_clusters` / `rank_schedule_max_rank` via the HiRef dynamic program.

Splits use vMF fitting on the unit sphere (cosine cost) with balanced or semi-relaxed *unbalanced* 
Sinkhorn assignments, and the HiRef `argmax` Assign rule. Only the first `supervised_depth` (early, stable) 
levels emit pseudo-labels; deeper levels are built toward the instance level but not supervised. 
See `.agents/designs/hierarchical_kway_vmf_ot_design.md`.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is only available inside Google Colab; skipping Drive mount.')

In [ ]:
import getpass
import os
import subprocess
from pathlib import Path
from urllib.parse import quote

PUBLIC_REPO_URL = 'https://github.com/moucheng2017/my_simsiam.git'
BRANCH = '61-recursive-ot'
REPO_DIR = Path('/content/my_simsiam')
GITHUB_TOKEN = ""  # Set to a token string here if the repo is private.


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    pass

repo_url = PUBLIC_REPO_URL
if GITHUB_TOKEN:
    repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(GITHUB_TOKEN, safe="")}@')


def _prompt_token():
    """Prompt for a GitHub token, guarding against Colab's patched getpass returning a dict."""
    try:
        from google.colab import userdata as _ud
        tok = _ud.get('GITHUB_TOKEN') or ''
        if tok:
            return tok
    except Exception:
        pass
    raw = getpass.getpass('GitHub token (leave blank to stop): ')
    return raw.strip() if isinstance(raw, str) else ''


def sync_repo():
    if GITHUB_TOKEN:
        run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_DIR)
    else:
        print('No GitHub token found; keeping the existing origin URL for fetch/pull.')
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)


os.chdir('/content')

if not REPO_DIR.exists():
    try:
        run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
    except subprocess.CalledProcessError as exc:
        if not GITHUB_TOKEN:
            print('\nClone failed without a token. If this repository is private, add a GITHUB_TOKEN Colab secret or paste a token below.')
            token = _prompt_token()
            if token:
                repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(token, safe="")}@')
                run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
            else:
                raise RuntimeError(
                    'Repository clone was skipped because no token was provided. '
                    'Add a GITHUB_TOKEN secret in Colab (Secrets panel) and re-run.'
                ) from exc
        else:
            raise
else:
    print(f'Repo already exists at {REPO_DIR}; pulling latest changes from {BRANCH}.')
    try:
        sync_repo()
    except subprocess.CalledProcessError as exc:
        raise RuntimeError(
            'Failed to fetch the latest repo changes. If the repository is private, make sure a GITHUB_TOKEN Colab secret is set or rerun after deleting /content/my_simsiam so the clone step can prompt for a token.'
        ) from exc

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Expected repository at {REPO_DIR}, but it was not created.')

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
run(['pip', 'install', '-r', 'requirements_colab.txt'], cwd=REPO_DIR)

In [ ]:
# CIFAR-10 is downloaded automatically by torchvision the first time this cell runs.
# The dataset is ~170 MB and will be saved to Google Drive so it only downloads once.
import os
import torchvision

DATA_DIR = '/content/drive/MyDrive/SSL_exps/data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Checking / downloading CIFAR-10 ...')
torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
print('CIFAR-10 is ready.')

In [ ]:
from colab_utils import train_from_colab
import math

# Resume / experiment identity
checkpoint_resume = None  # Set to a checkpoint .pth path to initialize the backbone.
pool_seed = 0

# Dataset / batch schedule
pool_size = 50000  # CIFAR-10 train pool size.
batch_size = 1024
iteration = int(pool_size // batch_size)  # Number of training batches sampled per epoch.

# Backbone embedding
embedding_dim = 256

# --- HiRef rank-annealing prior -------------------------------------------
# Per-level branching factors. Set to None to derive the optimal schedule from
# num_leaf_clusters / rank_schedule_max_rank via the HiRef dynamic program.
rank_schedule = [2, 4, 4, 8]  # e.g. None -> DP picks it; product = num_leaf_clusters.
num_leaf_clusters = 256       # DP target leaf count (used only when rank_schedule is None).
rank_schedule_depth = 4       # DP hierarchy depth (user budget); used only when rank_schedule is None.
rank_schedule_max_rank = 16   # DP per-level cap r_t <= this.
rank_schedule_base_rank = 1   # HiRef base rank Q (leaf block size); DP factors n/Q.
# Only the first supervised_depth (early, stable) levels emit pseudo-labels.
supervised_depth = 3

# --- vMF / OT fitting ------------------------------------------------------
kappa = 12.0       # Prediction-logit sharpness (cosine concentration).
ot_epsilon = 0.1   # Sinkhorn/OT entropy temperature.
sinkhorn_iters = 15  # K-way splits need more iterations than binary to balance.
em_iters = 5         # Alternating Sinkhorn assignment + vMF mean update steps.
batch_self_labeling = True  # Build the K-way tree inside each mini-batch.

# Prototype tree / memory
learnable_vmf_prototypes = False  # If True, prototypes are nn.Parameter values.
prototype_ema_momentum = 0.9      # None disables EMA; must be None if learnable.
tree_refresh_interval = 0         # >0 only for full-pool (batch_self_labeling=False).
tree_refresh_batch_size = 4096

# Depth annealing (staircase), clamped to supervised_depth. None disables it.
depth_annealing_epochs_per_level = None
depth_annealing_initial_depth = 1

# Unbalanced OT (semi-relaxed Sinkhorn). tau shares ot_epsilon's units; the
# trainer cosine-anneals it from nearly balanced toward natural split ratios.
ot_unbalanced_tau = 5.0
ot_unbalanced_min_split_fraction = 0.05  # collapse guard for hard K-way splits.
unbalanced_tau_start = 5.0
unbalanced_tau_final = 0.02
unbalanced_tau_anneal_epochs = 400

# Auxiliary sigmoid image-index regularization (0.0 disables it).
sigmoid_regularization_weight = 0.0
sigmoid_regularization_rampup_epochs = 50
sigmoid_init_temperature = 1.0
sigmoid_init_bias = -math.log(batch_size - 1)

# Purity/NMI tree diagnostics (per level, vs true labels; diagnostic only).
tree_metrics_interval = 1       # epochs between measurements; 0 disables.
tree_metrics_max_samples = 10000

# Optimization
base_lr = 0.0005  # The trainer scales this by batch_size / 256.
effective_base_lr = base_lr * batch_size / 256
warmup_epochs = 20
num_epochs = 400
stop_at_epoch = 400
checkpoint_saving_frequency = 100
optimizer_name = 'adam'
weight_decay = 0.0
disable_weight_decay = False
beta1 = 0.9
beta2 = 0.95

sched_tag = 'dp' if rank_schedule is None else '-'.join(str(r) for r in rank_schedule)
resume_tag = checkpoint_resume.split('/')[-1].replace('.pth', '') if checkpoint_resume else 'no'
exp_tag = (
    f'kway_p{pool_size}_bs{batch_size}_d{embedding_dim}_'
    f'sched{sched_tag}_sup{supervised_depth}_k{kappa:g}_eps{ot_epsilon:g}_em{em_iters}_'
    f'ema{prototype_ema_momentum}_'
    f'uot{unbalanced_tau_start:g}-{unbalanced_tau_final:g}-{unbalanced_tau_anneal_epochs}_'
    f'iter{iteration}-seed{pool_seed}-resume-{resume_tag}'
)
print(f'Using base_lr={base_lr} with batch_size={batch_size}; effective optimizer LR={effective_base_lr:g}')

hierarchical_kway_result = train_from_colab(
    config_file='configs/hierarchical_kway_vmf_ot_cifar_colab.yaml',
    project_name='SSL_exps',
    use_drive=True,
    device='cuda',
    download=False,  # CIFAR-10 is pre-downloaded by the cell above.
    overrides={
        'name': exp_tag,
        'checkpoint_resume': checkpoint_resume,
        'model': {
            'embedding_dim': embedding_dim,
            'rank_schedule': rank_schedule,
            'num_leaf_clusters': num_leaf_clusters,
            'rank_schedule_depth': rank_schedule_depth,
            'rank_schedule_max_rank': rank_schedule_max_rank,
            'rank_schedule_base_rank': rank_schedule_base_rank,
            'supervised_depth': supervised_depth,
            'kappa': kappa,
            'ot_epsilon': ot_epsilon,
            'sinkhorn_iters': sinkhorn_iters,
            'em_iters': em_iters,
            'batch_self_labeling': batch_self_labeling,
            'learnable_vmf_prototypes': learnable_vmf_prototypes,
            'prototype_ema_momentum': prototype_ema_momentum,
            'ot_unbalanced_tau': ot_unbalanced_tau,
            'ot_unbalanced_min_split_fraction': ot_unbalanced_min_split_fraction,
            'sigmoid_regularization_weight': sigmoid_regularization_weight,
            'sigmoid_init_temperature': sigmoid_init_temperature,
            'sigmoid_init_bias': sigmoid_init_bias,
        },
        'train': {
            'batch_size': batch_size,
            'source_pool_size': pool_size,
            'source_subset_seed': pool_seed,
            'num_epochs': num_epochs,
            'stop_at_epoch': stop_at_epoch,
            'base_lr': base_lr,
            'samples_per_epoch': batch_size * iteration,
            'warmup_epochs': warmup_epochs,
            'checkpoint_saving_frequency': checkpoint_saving_frequency,
            'tree_refresh_interval': tree_refresh_interval,
            'tree_refresh_batch_size': tree_refresh_batch_size,
            'depth_annealing_epochs_per_level': depth_annealing_epochs_per_level,
            'depth_annealing_initial_depth': depth_annealing_initial_depth,
            'unbalanced_tau_start': unbalanced_tau_start,
            'unbalanced_tau_final': unbalanced_tau_final,
            'unbalanced_tau_anneal_epochs': unbalanced_tau_anneal_epochs,
            'sigmoid_regularization_rampup_epochs': sigmoid_regularization_rampup_epochs,
            'tree_metrics_interval': tree_metrics_interval,
            'tree_metrics_max_samples': tree_metrics_max_samples,
            'optimizer': {
                'name': optimizer_name,
                'weight_decay': weight_decay,
                'disable_weight_decay': disable_weight_decay,
                'beta1': beta1,
                'beta2': beta2,
            },
        }
    },
)
hierarchical_kway_result